

#Topic Modeling in NLP --- >

- 1 paragraph(corpus)-->1 sentence(document)--> 1documents----> 1 topic identify
- Topic Modeling is an unsupervised technique that discovers hidden themes/topics from a large collection of documents - without any
manual labeling.
- Simple example: If you have 1000 news articles, a topic model will automatically figure out which articles are about "Sports," which are
about "Politics," which are about "Tech"- without anyone labeling them.

**How it works (concept level):**
- Each document is treated as a "mixture of topics" Each topic is treated as a "mixture of words" The algorithm statistically figures out
which words frequently occur together across documents - that becomes a topic.


# LDA (Latent Dirichlet Allocation):- LDA is a method to find hidden topics inside text.

 **How it works →**

- Breaks documents into words (tokenize).
- Removes common words (stopwords like is, the, and).
- Makes a Bag of Words (BoW) → counts of each word.
- LDA looks at these counts and groups words into topics.


**Uses of LDA**
- Summarization → Quickly summarize large text collections.
- Clustering → Group similar documents together.
- Search engines → Improve indexing by topics.
- Recommendation systems → Suggest content based on topics.
- Text analysis → Discover themes in articles, research papers, or social media posts.
            

**Advantages**
- **Unsupervised →** No need for labeled data.
- **Scalable →** Works on large text datasets.
- **Interpretable →** Topics are human-readable (lists of words).
- **Flexible →** Can be applied to many domains (news, research, customer feedback).

**Limitations**
- Static topics → Topics don’t change with context.
- **Quality depends on preprocessing →** Stopword removal, tokenization, etc. must be done carefully.
- **Number of topics (k) →** Must be chosen manually, which can be tricky.
- **Polysemy issue →** Same word with different meanings (e.g., bank) may confuse the model.

```
Document (Raw Text)
        │
        ▼
Preprocessing
   - Lowercasing
   - Remove stopwords (NLTK)
   - Tokenize (word_tokenize)
        │
        ▼
Bag of Words (BoW)
   - Dictionary (word → ID)
   - Corpus (list of (ID, freq))
        │
        ▼
LDA Model (gensim.models.LdaModel)
   - num_topics = k
   - passes = iterations
   - alpha, beta = priors
        │
        ▼
Output
   - Topic distribution per doc
   - Top words per topic
```


In [ ]:
#!pip install gensim pyLDAvis nltk -q

In [ ]:
import pandas as pd
import gensim
from gensim import corpora
from gensim.models import LdaModel
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
import string
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
#load csv here: df=pd.read_csv("news.csv")
documents = [
    "The government announced new tax policies for small businesses",
    "The cricket team won the match with a great century",
    "New AI model beats previous benchmarks in language understanding",
    "Election results show major shift in voter sentiment",
    "The football league started with an exciting opening match",
    "Machine learning models are being deployed in healthcare systems",
    "Parliament passed a new bill regarding education reforms",
    "The basketball tournament final was won by the home team",
    "Deep learning research is advancing rapidly across the industry",
    "Opposition party criticized the government's new policy decision"
]


In [ ]:
#text preprocessing
stop_words = set(stopwords.words('english'))

def preprocess(text):
  text=text.lower()
  tokens=word_tokenize(text)
  tokens=[t for t in tokens if t not  in string.punctuation]
  tokens=[t for t in tokens if t not in stop_words and t.isalpha()]
  return tokens

processed_docs=[preprocess(doc) for doc in documents]
print(processed_docs[0])  #check output


['government', 'announced', 'new', 'tax', 'policies', 'small', 'businesses']


In [ ]:
#create Dictionary @corpus
dictionary=corpora.Dictionary(processed_docs)
#optional: remove rare/very common words
dictionary.filter_extremes(no_below=2,no_above=0.8)
#create bag of words
corpus=[dictionary.doc2bow(doc) for doc in processed_docs]


In [ ]:
from IPython.utils.sysinfo import num_cpus
#train LDA model
NUM_TOPICS=3 #tune this based on your data
lda_model=LdaModel(corpus=corpus, #numeric (id,count) format of all documents
                   id2word=dictionary, #ID-> word mapping(so model output show actual words,not numbers)
                   num_topics=NUM_TOPICS, #how many toipcs to discovers(3)
                   random_state=100, #for reproducibility - same seed = same result every run
                   alpha='auto', #Document-topic distribution ko automatically adjust karega.
                   passes=10, # how many time training
                   per_word_topics=True #Har word ke liye topic distribution bhi nikalta hai (extra detail).
                   )

In [ ]:
#print the topic
for idx,topic in lda_model.print_topics(num_words=6):
  print(f"Topic {idx}: {topic}")
  print()

Topic 0: 0.625*"new" + 0.095*"learning" + 0.094*"match" + 0.094*"team" + 0.093*"government"

Topic 1: 0.417*"new" + 0.405*"government" + 0.060*"learning" + 0.059*"team" + 0.059*"match"

Topic 2: 0.304*"team" + 0.304*"match" + 0.303*"learning" + 0.045*"new" + 0.044*"government"



In [ ]:
#Assign topics to documents
for i,doc_bow in enumerate(corpus):
  topics=lda_model.get_document_topics(doc_bow)
  dominant_topic=max(topics,key=lambda x:x[1])
  print(f"Doc {i}: '{documents[i][:50]}...' -> topic: {dominant_topic[0]} (score: {dominant_topic[1]:.2f})")


Doc 0: 'The government announced new tax policies for smal...' -> topic: 1 (score: 0.70)
Doc 1: 'The cricket team won the match with a great centur...' -> topic: 2 (score: 0.78)
Doc 2: 'New AI model beats previous benchmarks in language...' -> topic: 0 (score: 0.53)
Doc 3: 'Election results show major shift in voter sentime...' -> topic: 2 (score: 0.46)
Doc 4: 'The football league started with an exciting openi...' -> topic: 2 (score: 0.69)
Doc 5: 'Machine learning models are being deployed in heal...' -> topic: 2 (score: 0.69)
Doc 6: 'Parliament passed a new bill regarding education r...' -> topic: 0 (score: 0.53)
Doc 7: 'The basketball tournament final was won by the hom...' -> topic: 2 (score: 0.69)
Doc 8: 'Deep learning research is advancing rapidly across...' -> topic: 2 (score: 0.69)
Doc 9: 'Opposition party criticized the government's new p...' -> topic: 1 (score: 0.70)
